# Team comparison export

Formats the regressor's and classifier's results for the team's shared `Resultados_modelos.xlsx`.
Loads predictions already saved in Días 4-5, no retraining. `Global` pools all 3 folds' test
predictions into one dataset, not an average of the fold-level numbers.

In [1]:
# Jupyter's cwd is the notebook's folder, not the repo root, so "from src.common..." below
# needs the repo root on sys.path. This walks up parent folders until it finds requirements.txt
# (which only exists at the repo root) and adds that folder to sys.path.
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "requirements.txt").exists():
            return parent
    raise FileNotFoundError("Could not locate repo root (requirements.txt not found)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

WindowsPath('C:/workspace/CAPSTONE/ontario-electricity-peak-risk')

In [2]:
import pandas as pd

from src.common.data_loading import INTERIM_DIR
from src.common.metrics import classification_summary_metrics, summary_metrics

pd.set_option("display.max_columns", 50)

FOLD_YEARS = {1: 2023, 2: 2024, 3: 2025}

## 1. Load saved predictions

Both files are the long-format outputs saved at the end of Días 4-5: one row per prediction, a
`fold` column marking which fold's test split it came from.

In [3]:
regressor_predictions = pd.read_parquet(INTERIM_DIR / "lightgbm_regressor_predictions.parquet")
classifier_predictions = pd.read_parquet(INTERIM_DIR / "lightgbm_classifier_predictions.parquet")
regressor_predictions.shape, classifier_predictions.shape

((3787776, 8), (3785976, 9))

## 2. Regressor: Global + per-fold, Excel columns

`Global` is the full table (folds already concatenated when saved); each fold row just filters
to that fold. WAPE is x100 to match the sheet's convention (Jorge's row 4); MAE/RMSE/BIAS stay
in kWh.

In [4]:
def regressor_row(df):
    metrics = summary_metrics(df)
    return {
        "MAE": round(metrics["mae"], 2),
        "RMSE": round(metrics["rmse"], 2),
        "MAPE": round(metrics["mape"], 2),
        "WAPE": round(metrics["wape"] * 100, 2),
        "BIAS": round(metrics["bias"], 2),
    }

regressor_table = {"Global": regressor_row(regressor_predictions)}
for fold_number, year in FOLD_YEARS.items():
    regressor_table[f"Fold {year}"] = regressor_row(
        regressor_predictions[regressor_predictions["fold"] == fold_number]
    )

regressor_export = pd.DataFrame(regressor_table).T
regressor_export

,MAE,RMSE,MAPE,WAPE,BIAS
Global,561.63,942.60,5.96,6.05,-55.93
Fold 2023,612.74,1050.38,6.47,6.79,-127.57
Fold 2024,517.28,823.05,5.70,5.64,-18.23
Fold 2025,554.99,940.93,5.72,5.74,-22.06


## 3. Classifier: Global + per-fold, Excel columns

Same idea. Everything here already comes out of `classification_summary_metrics` in the units
the sheet expects, no extra scaling needed.

In [5]:
def classifier_row(df):
    metrics = classification_summary_metrics(df)
    return {
        "precision": round(metrics["precision"], 4),
        "recall": round(metrics["recall"], 4),
        "f1": round(metrics["f1"], 4),
        "balanced_accuracy": round(metrics["balanced_accuracy"], 4),
        "positive_rate_pct": round(metrics["positive_rate_pct"], 2),
        "pr_auc": round(metrics["pr_auc"], 4),
        "roc_auc": round(metrics["roc_auc"], 4),
        "brier": round(metrics["brier"], 4),
    }

classifier_table = {"Global": classifier_row(classifier_predictions)}
for fold_number, year in FOLD_YEARS.items():
    classifier_table[f"Fold {year}"] = classifier_row(
        classifier_predictions[classifier_predictions["fold"] == fold_number]
    )

classifier_export = pd.DataFrame(classifier_table).T
classifier_export

,precision,recall,f1,balanced_accuracy,positive_rate_pct,pr_auc,roc_auc,brier
Global,0.9450,0.9165,0.9306,0.9562,7.30,0.9509,0.9843,0.0088
Fold 2023,0.9782,0.8634,0.9172,0.9305,10.96,0.9335,0.9747,0.0157
Fold 2024,0.8864,0.9889,0.9348,0.9918,4.00,0.9932,0.9995,0.0046
Fold 2025,0.9368,0.9587,0.9476,0.9769,6.94,0.9841,0.9971,0.0063


## 4. Save as CSV for reference

Not part of the pipeline, just a copy-paste-friendly backup for rows 7/15 in case the shared
Excel is being edited by someone else.

In [6]:
export_dir = REPO_ROOT / "notebooks" / "04_evaluation"
regressor_export.to_csv(export_dir / "regressor_export.csv")
classifier_export.to_csv(export_dir / "classifier_export.csv")
list(export_dir.glob("*.csv"))

[WindowsPath('C:/workspace/CAPSTONE/ontario-electricity-peak-risk/notebooks/04_evaluation/classifier_export.csv'),
 WindowsPath('C:/workspace/CAPSTONE/ontario-electricity-peak-risk/notebooks/04_evaluation/regressor_export.csv')]